THE integration
raw data -> csv, APIs, Dataframes

data engineering -> ETL, Cleaning, Pandas, SQL

data storage -> SQLite, csv, VectorDB

genai layer -> Groq API,RAG Agents
ai application Chatbot ,

Layer 1-> data integration

Layer 2-> Data Processing

Layer 3->AI reasoning

Layer 4->Delivery+Interface

# Capstone Project

1. AI Data Analyst :
Analyse student performance data using Pandas ,SQL,and Groq AI. Answer natural language question about the data automatically.

2. College Knowledge Assistant:
Build a RAG system that answer questions about your college notes using ChromaDB embeddings and the Groq LLM.

3. AI Data Cleaner
Build an intelligent data cleaning agent that detects problems in datasets and fixes them automatically using AI prompts.

In [12]:
!pip install groq chromadb sentence-transformers pandas -q
print("All packages installed succesfuuly.")
print("Ready for Day 10 - Internship finale!")

All packages installed succesfuuly.
Ready for Day 10 - Internship finale!


In [8]:
import pandas as pd
import sqlite3
from groq import Groq
import chromadb
import os
import json
print("All imports successfully")
print("Libraries loaded: pandas, sqlite3,groq, chromadb, os,json")

All imports successfully
Libraries loaded: pandas, sqlite3,groq, chromadb, os,json


In [9]:
GROQ_API_KEY = "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"
client = Groq(api_key=GROQ_API_KEY)
MODEL = "llma-3.1-8b-instant"
print("Groq client configured.")
print(f"Model:{MODEL}")
print("Status:Ready to generate AI responses")

Groq client configured.
Model:llma-3.1-8b-instant
Status:Ready to generate AI responses


In [10]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

MODEL = "llama-3.3-70b-versatile"  # use a model from your account

responsible_system_prompt = """
You are a helpful data analysis assistant.
You only answer questions based on the data provided to you.
If you are not sure about something, say:
'I do not have enough information to answer that accurately.'
Never make up statistics or facts that are not in the data you receive.
"""

test_response = client.chat.completions.create(
    model=MODEL,
    max_tokens=100,
    messages=[
        {"role": "system", "content": responsible_system_prompt},
        {"role": "user", "content": "What is the population of Mars?"}
    ]
)

print("=== Responsible AI Test ===")
print("Question: What is the population of Mars?")
print("AI Response:")
print(test_response.choices[0].message.content)

=== Responsible AI Test ===
Question: What is the population of Mars?
AI Response:
I do not have enough information to answer that accurately.


In [11]:
student_df = pd.read_csv("student_performance.csv")
print(f"=== Student Performance Dataset ===")
print(f"Shape: {student_df.shape[0]} students, {student_df.shape[1]} columns")
print()
print(student_df.head(5))

=== Student Performance Dataset ===
Shape: 30 students, 11 columns

   student_id          name  age  gender branch  attendance_pct  \
0           1  Aarav Sharma   20    Male    CSE              85   
1           2   Priya Patel   21  Female    ECE              92   
2           3   Rohit Kumar   20    Male   MECH              67   
3           4    Sneha Iyer   22  Female    CSE              95   
4           5  Vikram Singh   21    Male  CIVIL              72   

   assignment_score  midterm_score  final_score  gpa passed  
0                78             72           76  7.6    Yes  
1                88             85           89  8.9    Yes  
2                55             60           58  5.8    Yes  
3                92             90           94  9.4    Yes  
4                62             65           63  6.3    Yes  


In [15]:
notes_df = pd.read_csv("college_notes.csv")
print("=== College Notes Dataset ===")
print(f"Shape:{notes_df.shape[0]} notes, {notes_df.shape[1]} columns")
print()
print(notes_df[['note_id','subject','topic','difficulty']].to_string(index=False))

=== College Notes Dataset ===
Shape:15 notes, 6 columns

 note_id             subject                       topic   difficulty
       1     Data Structures                      Arrays     Beginner
       2     Data Structures                Linked Lists     Beginner
       3     Data Structures                Binary Trees Intermediate
       4     Data Structures           Stacks and Queues     Beginner
       5 Database Management                  SQL Basics     Beginner
       6 Database Management               Normalization Intermediate
       7 Database Management                    Indexing Intermediate
       8    Machine Learning                  Regression Intermediate
       9    Machine Learning              Classification Intermediate
      10    Machine Learning                  Clustering     Advanced
      11  Python Programming                   Functions     Beginner
      12  Python Programming Object Oriented Programming Intermediate
      13  Python Programming     

In [16]:
print("=== Data Quality Report:  student_performance.csv ===")
print("Missing values per column:")
print(student_df.head().sum())
print()
print("Data Types:")
print(student_df.dtypes)
print()

print(f"Duplicates row: {student_df.duplicated().sum()}")
print("Data Quality check complete.")

=== Data Quality Report:  student_performance.csv ===
Missing values per column:
student_id                                                         15
name                Aarav SharmaPriya PatelRohit KumarSneha IyerVi...
age                                                               104
gender                                       MaleFemaleMaleFemaleMale
branch                                             CSEECEMECHCSECIVIL
attendance_pct                                                    411
assignment_score                                                  375
midterm_score                                                     372
final_score                                                       380
gpa                                                              38.0
passed                                                YesYesYesYesYes
dtype: object

Data Types:
student_id            int64
name                 object
age                   int64
gender               object
branch    

In [17]:
conn = sqlite3.connect(":memory:")
student_df.to_sql('students',conn,if_exists ='replace',index=False)
print("SQL database created.")
print("Table 'students' loaded with", len(student_df),"rows.")

SQL database created.
Table 'students' loaded with 30 rows.


In [18]:
query1 = """
SELECT
   branch,
   COUNT(*) AS total_students,
   ROUND(AVG(gpa), 2) AS avg_gpa,
   ROUND(AVG(attendance_pct), 1) AS avg_attendance
FROM students
GROUP BY branch
ORDER BY avg_gpa DESC
"""

branch_analysis = pd.read_sql(query1, conn)

print("=== Branch-wise Performance Analysis ===")
print(branch_analysis.to_string(index=False))

=== Branch-wise Performance Analysis ===
branch  total_students  avg_gpa  avg_attendance
    IT               5     8.64            89.4
   CSE              10     7.42            80.0
  MECH               5     7.22            79.4
 CIVIL               4     6.75            75.0
   ECE               6     6.38            69.2


In [19]:
query2 = """
SELECT name,branch,gpa,attendance_pct,passed
FROM students
ORDER BY gpa DESC
LIMIT 5
"""
top_students = pd.read_sql(query2, conn)

print("=== Top 5 Students by GPA ===")
print(top_students.to_string(index=False))

=== Top 5 Students by GPA ===
            name branch  gpa  attendance_pct passed
  Meera Krishnan     IT  9.5              96    Yes
      Sneha Iyer    CSE  9.4              95    Yes
Lakshmi Chandran    CSE  9.2              94    Yes
    Swathi Menon     IT  9.1              93    Yes
     Priya Patel    ECE  8.9              92    Yes


In [21]:
query3 = """
SELECT
   passed,
   COUNT(*) AS student_count,
   ROUND(AVG(gpa), 2) AS avg_gpa
FROM students
GROUP BY passed
"""

pass_fail = pd.read_sql(query3, conn)

print("=== Pass/Fail Analysis ===")
print(pass_fail.to_string(index=False))

total = len(student_df)
passed = len(student_df[student_df['passed'] == 'Yes'])

pass_rate = round((passed / total) * 100, 1)

print(f"\nOverall Pass Rate: {pass_rate}% ({passed}/{total} students)")

=== Pass/Fail Analysis ===
passed  student_count  avg_gpa
    No              3     4.47
   Yes             27     7.61

Overall Pass Rate: 90.0% (27/30 students)


In [22]:
branch_summary = ""

for _, row in branch_analysis.iterrows():
    branch_summary += (
        f"- {row['branch']}: {row['total_students']} students, "
        f"Avg GPA {row['avg_gpa']}, "
        f"Avg Attendance {row['avg_attendance']}%\n"
    )

top_summary = ""

for _, row in top_students.iterrows():
    top_summary += (
        f"- {row['name']} ({row['branch']}): GPA {row['gpa']}\n"
    )

data_summary = f"""
STUDENT PERFORMANCE DATA SUMMARY

Total Students: {total}
Overall Pass Rate: {pass_rate}%
Average GPA across all students: {round(student_df['gpa'].mean(), 2)}

Performance by Branch:
{branch_summary}

Top 5 Students by GPA:
{top_summary}
"""

print("Data summary prepared:")
print(data_summary)

Data summary prepared:

STUDENT PERFORMANCE DATA SUMMARY

Total Students: 30
Overall Pass Rate: 90.0%
Average GPA across all students: 7.29

Performance by Branch:
- IT: 5 students, Avg GPA 8.64, Avg Attendance 89.4%
- CSE: 10 students, Avg GPA 7.42, Avg Attendance 80.0%
- MECH: 5 students, Avg GPA 7.22, Avg Attendance 79.4%
- CIVIL: 4 students, Avg GPA 6.75, Avg Attendance 75.0%
- ECE: 6 students, Avg GPA 6.38, Avg Attendance 69.2%


Top 5 Students by GPA:
- Meera Krishnan (IT): GPA 9.5
- Sneha Iyer (CSE): GPA 9.4
- Lakshmi Chandran (CSE): GPA 9.2
- Swathi Menon (IT): GPA 9.1
- Priya Patel (ECE): GPA 8.9




In [23]:
system_prompt  = """
You are an expert academic data analyst working for an engineering college.
You receive student performance summaries and provide clear, actionable insights.
Always base your analysis strictly on the data provided.
If you are uncertain about something, say so clearly.
Format your response with numbered points for clarity
"""

In [24]:
user_message= f"""
Here is the student performance data for this semester:
{data_summary}
Please provide:
1.Three key insights from this data
2. Which branch needs the most improvement?
3. One recommendation for the college principal
"""

In [25]:
system_prompt  = """
You are an expert academic data analyst working for an engineering college.
You receive student performance summaries and provide clear, actionable insights.
Always base your analysis strictly on the data provided.
If you are uncertain about something, say so clearly.
Format your response with numbered points for clarity
"""
user_message= f"""
Here is the student performance data for this semester:
{data_summary}
Please provide:
1.Three key insights from this data
2. Which branch needs the most improvement?
3. One recommendation for the college principal
"""
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=600,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "system", "content": user_message}
    ]
)
al_analysis = response.choices[0].message.content
print("=" * 60)
print("AI-GENERATED DATA ANALYSIS")
print("=" * 60)
print(al_analysis)

AI-GENERATED DATA ANALYSIS
Here are my analysis and insights based on the provided data:

1. Three key insights from this data:
   * The overall pass rate of 90.0% suggests that the college is performing well in terms of student success.
   * The IT branch has the highest average GPA (8.64) and attendance rate (89.4%), indicating strong academic performance and engagement among IT students.
   * There is a noticeable variation in average GPA and attendance rates across different branches, suggesting that some branches may require targeted support or interventions to improve student outcomes.

2. The branch that needs the most improvement is ECE, with an average GPA of 6.38 and an average attendance rate of 69.2%, which are the lowest among all branches. This suggests that ECE students may be struggling academically and/or have lower engagement levels, warranting attention and support from college administrators and faculty.

3. One recommendation for the college principal:
   * Conside

In [26]:
system_prompt  = """
You are an expert academic data analyst working for an engineering college.
You receive student performance summaries and provide clear, actionable insights.
Always base your analysis strictly on the data provided.
If you are uncertain about something, say so clearly.
Format your response with numbered points for clarity
"""
user_message= f"""
Here is the student performance data for this semester:
{data_summary}
Please provide:
1.Three key insights from this data
2. Which branch needs the most improvement?
3. One recommendation for the college principal
"""
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=600,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "system", "content": user_message}
    ]
)
al_analysis = response.choices[0].message.content
print("=" * 60)
print("AI-GENERATED DATA ANALYSIS")
print("=" * 60)
print(al_analysis)

AI-GENERATED DATA ANALYSIS
Here are the analysis results based on the provided data:

1. **Three key insights from the data**:
    * The overall pass rate is 90.0%, indicating a high level of academic achievement across the college.
    * The IT branch has the highest average GPA (8.64) and attendance rate (89.4%), suggesting a strong performance by IT students.
    * There is a notable variation in average GPA across branches, with IT and CSE branches performing significantly better than MECH, CIVIL, and ECE branches.

2. **Branch needing the most improvement**:
    * Based on the data, the ECE branch appears to need the most improvement, with the lowest average GPA (6.38) and attendance rate (69.2%) among all branches.

3. **Recommendation for the college principal**:
    * Consider providing additional academic support and resources to the ECE branch, such as tutoring or mentoring programs, to help improve student performance and attendance rates, and to reduce the gap in average GP

**Capstone Project**

Project 1: AI Data Analyst

In [27]:
!pip install groq pandas matplotlib seaborn -q

In [28]:
import pandas as pd
import sqlite3
from groq import Groq

In [29]:
student_df = pd.DataFrame({
    "student_id":[1,2,3,4,5],
    "name":["Asha","Rahul","Priya","Kiran","Arun"],
    "subject":["Math","Science","Math","Science","Math"],
    "marks":[85,70,92,60,78]
})

student_df.to_csv("students.csv",index=False)

df = pd.read_csv("students.csv")
df.head()

,student_id,name,subject,marks
0,1,Asha,Math,85
1,2,Rahul,Science,70
2,3,Priya,Math,92
3,4,Kiran,Science,60
4,5,Arun,Math,78


In [30]:
conn = sqlite3.connect("students.db")

df.to_sql(
    "students",
    conn,
    if_exists="replace",
    index=False
)

print("Database created")

Database created


In [33]:
!pip install groq -q

from groq import Groq

GROQ_API_KEY = "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"

client = Groq(api_key=GROQ_API_KEY)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role":"user","content":"What is Python?"}
    ]
)

print(response.choices[0].message.content)



**Python** is a high-level, interpreted programming language that is widely used for various purposes such as:

* **Web development**: building web applications and web services
* **Data analysis**: working with data, performing statistical analysis, and creating visualizations
* **Artificial intelligence**: building machine learning models, natural language processing, and computer vision applications
* **Automation**: automating tasks, scripts, and workflows
* **Scientific computing**: performing scientific simulations, data analysis, and visualization

Python is known for its:

* **Easy-to-learn syntax**: simple and intuitive syntax makes it accessible to beginners
* **Large community**: extensive community support and a vast collection of libraries and frameworks
* **Cross-platform compatibility**: can run on multiple operating systems, including Windows, macOS, and Linux
* **Extensive libraries**: wide range of libraries and frameworks that make it suitable for various application

In [34]:
def get_schema(conn):

    cursor = conn.cursor()

    cursor.execute("""
    SELECT sql
    FROM sqlite_master
    WHERE type='table'
    """)

    schema = "\n".join(
        row[0] for row in cursor.fetchall()
    )

    return schema

In [35]:
def generate_sql(question,schema):

    prompt=f"""
You are a SQL expert.

Schema:
{schema}

Convert this question into SQL:

{question}

Return only SQL.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"user","content":prompt}
        ]
    )

    return response.choices[0].message.content.strip()

In [36]:
def run_sql(query,conn):

    try:
        result = pd.read_sql(query,conn)
        return result

    except Exception as e:
        return str(e)

In [37]:
def ask_data(question):

    schema = get_schema(conn)

    sql_query = generate_sql(
        question,
        schema
    )

    print("Generated SQL:")
    print(sql_query)

    result = run_sql(sql_query,conn)

    print("\nResult:")
    print(result)

In [38]:
ask_data(
    "What is the average marks in each subject?"
)

Generated SQL:
```sql
SELECT subject, AVG(marks) AS average_marks
FROM students
GROUP BY subject;
```

Result:
Execution failed on sql '```sql
SELECT subject, AVG(marks) AS average_marks
FROM students
GROUP BY subject;
```': near "```sql
SELECT subject, AVG(marks) AS average_marks
FROM students
GROUP BY subject;
```": syntax error


Project 2: College Knowledge Assistant (RAG)

In [39]:
!pip install chromadb
!pip install sentence-transformers
!pip install groq

In [40]:
import pandas as pd

notes_df = pd.DataFrame({

    "id":[1,2,3],

    "subject":[
        "Data Warehouse",
        "Python",
        "DBMS"
    ],

    "topic":[
        "ETL",
        "Functions",
        "Normalization"
    ],

    "content":[
        "ETL stands for Extract Transform Load.",
        "Functions are reusable blocks of code.",
        "Normalization reduces redundancy."
    ]
})

notes_df.head()

,id,subject,topic,content
0,1,Data Warehouse,ETL,ETL stands for Extract Transform Load.
1,2,Python,Functions,Functions are reusable blocks of code.
2,3,DBMS,Normalization,Normalization reduces redundancy.


In [41]:
import chromadb

client_db = chromadb.Client()

collection = client_db.create_collection(
    name="college_notes"
)

In [42]:
all_documents = notes_df["content"].tolist()

all_ids = notes_df["id"].astype(str).tolist()

all_metadatas = [

    {
        "subject":row["subject"],
        "topic":row["topic"]
    }

    for _,row in notes_df.iterrows()
]

collection.add(
    documents=all_documents,
    ids=all_ids,
    metadatas=all_metadatas
)

print("Notes stored")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 43.3MiB/s]


Notes stored


In [43]:
def search_notes(query):

    results = collection.query(
        query_texts=[query],
        n_results=3
    )

    return results

In [44]:
from groq import Groq

groq_client = Groq(
    api_key=GROQ_API_KEY
)

In [45]:
def ask_notes(question):

    results = search_notes(question)

    docs = results["documents"][0]

    context = "\n".join(docs)

    prompt = f"""
Answer only from the context.

Context:
{context}

Question:
{question}
"""

    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return response.choices[0].message.content

In [46]:
ask_notes(
    "What is ETL?"
)

'ETL stands for Extract Transform Load.'

Project 3: AI Data Cleaner

In [48]:
!pip install pandas groq -q

In [49]:
import pandas as pd

df = pd.DataFrame({

    "Name":[
        "Asha",
        None,
        "Rahul",
        "Rahul"
    ],

    "Age":[
        20,
        21,
        None,
        None
    ],

    "Marks":[
        85,
        90,
        90,
        90
    ]
})

df

,Name,Age,Marks
0,Asha,20.0,85
1,None,21.0,90
2,Rahul,NaN,90
3,Rahul,NaN,90


In [50]:
def data_report(df):

    print("Shape:",df.shape)

    print("\nMissing Values")

    print(df.isnull().sum())

    print("\nDuplicates")

    print(df.duplicated().sum())

In [51]:
def ai_suggestions(df):

    report = f"""
Columns:
{df.columns.tolist()}

Missing:
{df.isnull().sum()}

Duplicates:
{df.duplicated().sum()}
"""

    prompt=f"""
You are a data cleaning expert.

Dataset Report:

{report}

Suggest cleaning actions.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return response.choices[0].message.content

In [52]:
def auto_clean(df):

    cleaned = df.copy()

    cleaned = cleaned.drop_duplicates()

    for col in cleaned.columns:

        if cleaned[col].dtype == "object":

            cleaned[col] = cleaned[col].fillna(
                cleaned[col].mode()[0]
            )

        else:

            cleaned[col] = cleaned[col].fillna(
                cleaned[col].median()
            )

    return cleaned

In [53]:
data_report(df)

print(ai_suggestions(df))

clean_df = auto_clean(df)

clean_df

Shape: (4, 3)

Missing Values
Name     1
Age      2
Marks    0
dtype: int64

Duplicates
1
**Data Cleaning Recommendations**

Based on the provided dataset report, the following cleaning actions are suggested:

### Handling Missing Values

1. **Name**: Since there's only 1 missing value, it might be possible to manually fill in the correct name if the data source is available. If not, it can be replaced with a placeholder (e.g., "Unknown") or removed from the dataset, depending on the analysis requirements.
2. **Age**: With 2 missing values, we have a few options:
	* If the age is not crucial for the analysis, we can remove these rows.
	* If age is important, we can try to impute the missing values using statistical methods like mean, median, or mode. Alternatively, we can use more advanced imputation techniques like regression imputation or K-Nearest Neighbors (KNN) imputation.

### Removing Duplicates

1. **Duplicates**: There is only 1 duplicate row, which can be safely removed to en

,Name,Age,Marks
0,Asha,20.0,85
1,Asha,21.0,90
2,Rahul,20.5,90
